# Fast Ray Images (Two Pooch Files)

This notebook fetches the two Zenodo sample files used in this repo and builds one ray-based image per file.

It uses `OctreeRayInterpolator` and picks the faster of:
- `integrate_field_along_rays` (exact segment integral)
- `integrate_field_along_rays_midpoint` (midpoint per segment)


In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pooch

from starwinds_readplt.dataset import Dataset

from batcamp import OctreeInterpolator
from batcamp import OctreeRayInterpolator


### Fetch The Two Pooch Files


In [ ]:
URL = 'https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz'
KNOWN_HASH = 'c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136'
MEMBERS = {
    'SC': 'run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt',
    'IH': 'run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt',
}

paths = {
    key: Path(
        pooch.retrieve(
            url=URL,
            known_hash=KNOWN_HASH,
            progressbar=False,
            processor=pooch.Untar(members=[member]),
        )[0]
    )
    for key, member in MEMBERS.items()
}

paths


### Ray Image Helpers

Camera: launch rays from the `xmin` side, direction `+x`.


In [ ]:
def make_x_camera(tree, *, ny=64, nz=64):
    dmin, dmax = tree.domain_bounds(coord='xyz')
    xmin, xmax = float(dmin[0]), float(dmax[0])
    ymin, ymax = float(dmin[1]), float(dmax[1])
    zmin, zmax = float(dmin[2]), float(dmax[2])

    y = np.linspace(ymin, ymax, ny)
    z = np.linspace(zmin, zmax, nz)
    Yg, Zg = np.meshgrid(y, z, indexing='xy')

    x0 = xmin + 1e-6 * (xmax - xmin)
    origins = np.column_stack(
        (
            np.full(Yg.size, x0, dtype=float),
            Yg.ravel(),
            Zg.ravel(),
        )
    )
    direction = np.array([1.0, 0.0, 0.0], dtype=float)
    t_end = (xmax - xmin) * 0.999999

    return origins, direction, t_end, (ymin, ymax, zmin, zmax), Yg.shape


def pick_fastest_integrator(ray, origins, direction, t_end, *, chunk_size=4096, n_bench=512):
    origins_bench = origins[: min(int(n_bench), int(origins.shape[0]))]

    methods = {
        'exact': lambda o: ray.integrate_field_along_rays(
            o,
            direction,
            0.0,
            float(t_end),
            chunk_size=chunk_size,
        ),
        'midpoint': lambda o: ray.integrate_field_along_rays_midpoint(
            o,
            direction,
            0.0,
            float(t_end),
            chunk_size=chunk_size,
        ),
    }

    timings = {}
    for name, fn in methods.items():
        _ = fn(origins_bench)  # warm
        t0 = time.perf_counter()
        _ = fn(origins_bench)
        timings[name] = time.perf_counter() - t0

    winner = min(timings, key=timings.get)
    return winner, methods[winner], timings


def render_ray_image(path, label, *, ny=64, nz=64, chunk_size=4096):
    ds = Dataset.from_file(str(path))
    interp = OctreeInterpolator(ds, ['Rho [g/cm^3]'])
    ray = OctreeRayInterpolator(interp)

    origins, direction, t_end, extent_xyz, image_shape = make_x_camera(interp.tree, ny=ny, nz=nz)

    method_name, method, timings = pick_fastest_integrator(
        ray,
        origins,
        direction,
        t_end,
        chunk_size=chunk_size,
    )

    t0 = time.perf_counter()
    values = np.asarray(method(origins), dtype=float).reshape(image_shape)
    elapsed = time.perf_counter() - t0

    finite_count = int(np.isfinite(values).sum())
    positive_count = int(np.count_nonzero(np.isfinite(values) & (values > 0.0)))

    print(
        f"{label}: tree_coord={interp.tree.tree_coord}, picked={method_name}, "
        f"bench(exact={timings['exact']:.3f}s, midpoint={timings['midpoint']:.3f}s), "
        f"full={elapsed:.3f}s, finite={finite_count}/{values.size}, "
        f"positive={positive_count}/{values.size}"
    )

    plot_data = np.where(
        np.isfinite(values) & (values > 0.0),
        np.log10(values),
        np.nan,
    )

    valid = np.isfinite(plot_data)
    if np.any(valid):
        vmin = float(np.nanpercentile(plot_data[valid], 2.0))
        vmax = float(np.nanpercentile(plot_data[valid], 98.0))
    else:
        vmin, vmax = -1.0, 1.0

    ymin, ymax, zmin, zmax = extent_xyz
    fig, ax = plt.subplots(figsize=(7.4, 6.2))
    cmap = plt.colormaps['viridis'].copy()
    cmap.set_bad('lightgray')

    im = ax.imshow(
        plot_data,
        origin='lower',
        extent=[ymin, ymax, zmin, zmax],
        aspect='equal',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel('Y [R]')
    ax.set_ylabel('Z [R]')
    ax.set_title(f"{label}: log10(Integral rho ds), method={method_name}")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label('log10(Integral rho ds)')
    fig.tight_layout()

    out_png = Path(f"ray_fast_{label.lower()}.png")
    fig.savefig(out_png, dpi=180, bbox_inches='tight')
    plt.show()

    return {
        'label': label,
        'tree_coord': str(interp.tree.tree_coord),
        'method': method_name,
        'timings': timings,
        'full_seconds': elapsed,
        'finite': finite_count,
        'positive': positive_count,
        'total': int(values.size),
        'png': str(out_png),
    }


### Render SC And IH Images


In [ ]:
results = []
for label in ('SC', 'IH'):
    results.append(render_ray_image(paths[label], label, ny=64, nz=64, chunk_size=4096))

results
